# 09 — Feature-group ablation

**Research question:** Which non-RNA tumor features provide predictive signal for ten TAA RNA-expression outcomes?

Separate MLPs are trained on five feature subsets using a consistent cohort-stratified 70/15/15 split, training-only preprocessing and fixed initialization seeds. Scores are associations, not causal attribution. **Caution:** previous ablation figures may have included project identifiers in modality-only features; do not compare results without rerunning.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
def root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/"config"/"genes.json").exists():return p
    raise FileNotFoundError("Repository root not found")
ROOT=root()
get_ipython().run_line_magic("run",str(ROOT/"notebooks"/"_Shared_Training_Utilities.ipynb"))
SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)
df=load_modeling_table().dropna(subset=TARGET_COLS).reset_index(drop=True)
features=get_feature_columns(df)
clinical=split_feature_modalities(features)["clinical"]
clinical=[x for x in clinical if x not in {"project_id_cn","project_id_mut"} and not x.endswith(("_mut","_cn"))]
cn=[f"{g}_cn" for g in GENE_CONFIG["copy_number_feature_genes"] if f"{g}_cn" in features]
mutation=[f"{g}_mut" for g in GENE_CONFIG["mutation_feature_genes"] if f"{g}_mut" in features]
sets={"Clinical/cancer type only":clinical,
      "Copy-number only":cn,
      "Mutation only":mutation,
      "Copy-number + mutation":cn+mutation,
      "All features":list(dict.fromkeys(clinical+cn+mutation))}
assert all(sets.values()),"Feature group missing: inspect modeling_table.csv"
assert all(not any(t in c.lower() for t in ("_rna_","_tpm","_expression")) for cs in sets.values() for c in cs)
for k,v in sets.items():print(k,len(v))

In [ ]:
train,val,test=make_splits(df,seed=SEED)
all_metrics=[]
all_hist=[]
for label,cols in sets.items():
    print("TRAIN:",label)
    pre,num,cat=build_preprocessor(train,cols)
    x_train=pre.fit_transform(prepare_feature_frame(train,num,cat))
    x_val=pre.transform(prepare_feature_frame(val,num,cat))
    x_test=pre.transform(prepare_feature_frame(test,num,cat))
    ys=StandardScaler()
    y_train=ys.fit_transform(train[TARGET_COLS].values)
    y_val=ys.transform(val[TARGET_COLS].values)
    torch.manual_seed(SEED) # seed BEFORE initializing weights
    model=StandardMLP(input_dim=x_train.shape[1],output_dim=len(TARGET_COLS),
                      hidden_dims=(256,128,64),dropout=0.40,batch_norm=True)
    model,hist,best_val,device,elapsed=train_torch_model(
        model,x_train,y_train,x_val,y_val,epochs=120,
        batch_size=64,patience=20,seed=SEED)
    model.eval()
    with torch.no_grad():
        p=model(torch.tensor(x_test,dtype=torch.float32).to(device)).cpu().numpy()
    y_pred=ys.inverse_transform(p)
    met=evaluate_predictions(test[TARGET_COLS].values,y_pred,TARGET_COLS)
    met["feature_group"]=label
    met["gene"]=met["target"].str.replace("_rna_log2tpm","",regex=False)
    met["input_feature_count"]=len(cols)
    met["best_val_loss"]=best_val
    all_metrics.append(met)
    hist["feature_group"]=label
    all_hist.append(hist)
    print("Mean held-out R²:",round(met.r2.mean(),3))
out=ROOT/"results"/"metrics"
out.mkdir(parents=True,exist_ok=True)
metrics=pd.concat(all_metrics,ignore_index=True)
metrics.to_csv(out/"feature_group_ablation_metrics.csv",index=False)
h=ROOT/"results"/"histories"
h.mkdir(parents=True,exist_ok=True)
pd.concat(all_hist,ignore_index=True).to_csv(h/"feature_group_ablation_history.csv",index=False)
display(metrics.pivot_table(index="gene",columns="feature_group",values="r2"))
